# 06 — Machine Unlearning (Smoothed GA)

Make the model 'forget' harmful or out-of-domain knowledge while preserving the retain set.

**Papers**
- Machine Unlearning in LLMs https://arxiv.org/abs/2405.15152
- Smoothed-GA https://openreview.net/forum?id=qd9fA4LzVN

In [ ]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])

from utils.colab import bootstrap
bootstrap()


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")


## 1. Setup & config

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, load_model_unsloth, apply_qlora, count_trainable_params,
    cuda_device_info,
)
from experiments.shared.eval_utils import compute_perplexity, benchmark_speed, vram_snapshot
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve


In [ ]:
set_env(quiet=True)
tracking, tags, run_name = setup_run(
    experiment='machine_unlearning',
    model=DEFAULT_MODEL_NAME,
    stage="experiment",
    mlflow_experiment_name='burnit-bg-experiments',
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")


## 2. Data loading

In [ ]:
# ###### Pick a dataset from MinIO (or fall back to local data_prep/) ######
# Set DATASET_PREFIX in .env / Colab secrets to skip the picker, e.g.
#   DATASET_PREFIX=data_prep/processed/mental-health
# Or pass `auto=True` below for non-interactive runs.

DEFAULT_PREFIX = os.getenv("DATASET_PREFIX", "data_prep/processed")
try:
    local_dataset_dir = resolve(prefix=DEFAULT_PREFIX, auto=False)
    TRAIN_PATH = local_dataset_dir / "train.jsonl"
    EVAL_PATH  = local_dataset_dir / "eval.jsonl"
except (FileNotFoundError, Exception) as exc:
    print(f"[data] MinIO lookup failed ({exc}); falling back to local data_prep/processed/")
    TRAIN_PATH = REPO_ROOT / "data_prep/processed/train.jsonl"
    EVAL_PATH  = REPO_ROOT / "data_prep/processed/eval.jsonl"

train_records = list(load_alpaca_dataset(TRAIN_PATH))
eval_records  = list(load_alpaca_dataset(EVAL_PATH))
train_stats = dataset_statistics(train_records)
eval_stats  = dataset_statistics(eval_records)
print(f"train: {len(train_records)} rows  eval: {len(eval_records)} rows")
print(train_stats)


## 2b. Forget vs retain splits

In [ ]:
forget_records = [r for r in train_records if r.get("is_refusal") or r.get("category") == "out_of_domain"]
retain_records = [r for r in train_records if r not in forget_records]
print(f"forget={len(forget_records)}  retain={len(retain_records)}")
tracking.log_metrics({
    "unlearn.forget_size": float(len(forget_records)),
    "unlearn.retain_size": float(len(retain_records)),
})


## 3. Smoothed gradient-ascent loop

In [ ]:
# Loss = -L_forget + λ1·L_retain + λ2·KL(model || frozen_reference)
# Forget loss ascends (negate); retain loss descends; KL anchor prevents
# catastrophic forgetting.
import torch, copy
LAMBDA_RETAIN = 0.5
LAMBDA_KL     = 0.3

reference = copy.deepcopy(model).eval()
for p in reference.parameters(): p.requires_grad_(False)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5)

def text_to_batch(records, n=4):
    import random
    batch = random.sample(records, k=min(n, len(records)))
    texts = [alpaca_to_prompt(r, eos_token=tokenizer.eos_token) for r in batch]
    return tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

EPOCHS = 1
STEPS_PER_EPOCH = 50
with tracking.run(run_name=run_name, tags=tags):
    for epoch in range(EPOCHS):
        for step in range(STEPS_PER_EPOCH):
            f_batch = text_to_batch(forget_records)
            r_batch = text_to_batch(retain_records)
            forget_out = model(**f_batch, labels=f_batch.input_ids)
            retain_out = model(**r_batch, labels=r_batch.input_ids)
            with torch.no_grad():
                ref_logits = reference(**r_batch).logits
            kl = torch.nn.functional.kl_div(
                torch.log_softmax(retain_out.logits, dim=-1),
                torch.softmax(ref_logits, dim=-1),
                reduction="batchmean",
            )
            total = -forget_out.loss + LAMBDA_RETAIN * retain_out.loss + LAMBDA_KL * kl
            total.backward(); optimizer.step(); optimizer.zero_grad()
            if step % 10 == 0:
                tracking.log_metrics({
                    "loss.forget":  float(forget_out.loss),
                    "loss.retain":  float(retain_out.loss),
                    "loss.kl":      float(kl),
                    "loss.total":   float(total),
                }, step=epoch * STEPS_PER_EPOCH + step)
                print(f"step={step}  forget={forget_out.loss:.3f} retain={retain_out.loss:.3f} kl={kl:.3f}")


## 4. Evaluate forget vs retain

In [ ]:
retain_ppl = compute_perplexity(model, tokenizer, [r["output"] for r in retain_records[:48]])
forget_ppl = compute_perplexity(model, tokenizer, [r["output"] for r in forget_records[:48]])
tracking.log_metrics({
    "unlearn.retain_ppl": float(retain_ppl),
    "unlearn.forget_ppl": float(forget_ppl),
    "unlearn.ppl_gap":    float(forget_ppl - retain_ppl),
})
print(f"retain ppl={retain_ppl:.2f}  forget ppl={forget_ppl:.2f}  (forget should be HIGHER)")


## 6. Inference test

In [ ]:
# ###### Inference test (mental-health prompts) ######
with stage(tracking, "inference_test"):
    batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
    log_responses(
        tracking,
        experiment='machine_unlearning',
        model_checkpoint=str(REPO_ROOT / "tmp/experiments/machine_unlearning"),
        **batteries,
    )
    for k, v in batteries.items():
        print(f"-- {k} --")
        for entry in v[:2]:
            print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")


## 7. Save via MLflow + GGUF export

In [ ]:
# ###### Save model via MLflow (single source of truth) ######
# tracking.log_model logs the model artifact under runs:/<id>/model and,
# when registered_model_name is set, adds a new version to the Models tab.
# MLflow's artifact store backs onto MinIO — no separate upload needed.
with stage(tracking, "save_model"):
    try:
        tracking.log_model(
            model,
            flavor="transformers",
            artifact_path="model",
            registered_model_name='burnit-bg-machine-unlearning',
            input_example=None,
        )
        print("[save] model logged via MLflow")
    except Exception as exc:
        print(f"[save] log_model failed: {exc}")

# Optional: GGUF export for offline local inference (RTX 3050 / Ollama).
# The GGUF is added as a run artifact under `gguf/`.
with stage(tracking, "gguf_export"):
    try:
        from experiments.shared.model_utils import export_gguf
        gguf_path = export_gguf(model, tokenizer, REPO_ROOT / "tmp/experiments/machine_unlearning/gguf", quantization="q4_k_m")
        tracking.save_data(gguf_path, artifact_path="gguf")
        print(f"[save] GGUF logged: {gguf_path}")
    except Exception as exc:
        print(f"[save] GGUF export skipped: {exc}")
